In [23]:
import requests
import pandas as pd
import matplotlib.pyplot as plt

# API endpoint URLs
emission_url  = "https://data.fingrid.fi/api/datasets/266/data"
wind_power_url   = "https://data.fingrid.fi/api/datasets/181/data"

# API keys
api_key_emission = "your_api_key"
api_key_wind = "your_api_key"

# Headers for the requests
headers_emission = {"x-api-key": api_key_emission}
headers_wind = {"x-api-key": api_key_wind}

# Fetching emission factor data
emission_response = requests.get(emission_url, headers=headers_emission)
wind_response = requests.get(wind_power_url, headers=headers_wind)

if emission_response.status_code == 200 and wind_response.status_code == 200:
    emission_data = emission_response.json()['data']
    wind_data = wind_response.json()['data']

    # Parse emission data into DataFrame
    emission_df = pd.DataFrame(emission_data)
    emission_df['startTime'] = pd.to_datetime(emission_df['startTime'])
    emission_df = emission_df[['startTime', 'value']].rename(columns={'value': 'emission_factor'})

    # Parse wind power data into DataFrame
    wind_df = pd.DataFrame(wind_data)
    wind_df['startTime'] = pd.to_datetime(wind_df['startTime'])
    wind_df = wind_df[['startTime', 'value']].rename(columns={'value': 'wind_power'})

    # Merge DataFrames on startTime
    combined_df = pd.merge(emission_df, wind_df, on='startTime', how='inner')

    # Plotting the data
    plt.figure(figsize=(12, 6))
    plt.plot(combined_df['startTime'], combined_df['emission_factor'], label='Emission Factor (g CO₂/kWh)')
    plt.plot(combined_df['startTime'], combined_df['wind_power'], label='Wind Power Production (MW)')
    plt.legend()
    plt.title('Emission Factor and Wind Power Production Over Time')
    plt.xlabel('Timestamp')
    plt.xticks(rotation=45)
    plt.show()

    # Correlation analysis
    correlation = combined_df['emission_factor'].corr(combined_df['wind_power'])
    print(f"Correlation between emission factor and wind power production: {correlation:.2f}")

else:
    print(f"Failed to fetch data: Emission Status {emission_response.status_code}, Wind Status {wind_response.status_code}")

Failed to fetch data: Emission Status 401, Wind Status 401
